In [ ]:
import os
import re
import glob
import random
import warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from lightgbm import LGBMClassifier, early_stopping

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

warnings.filterwarnings("ignore")

# =====================================================================
# 0. CONFIGURATION & SEEDING
# =====================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running V5 Spatial-Preserving + Transductive Self-Calibration Pipeline on: {DEVICE}")

EMG_COLS = [f'ch{i}' for i in range(1, 9)]
IMU_COLS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
SIGNAL_COLS = EMG_COLS + IMU_COLS

AUX_REG_COLS = [
    'roll', 'pitch', 'yaw', 'index_curl', 'middle_curl',
    'ring_curl', 'pinky_curl', 'pinch', 'openness', 'thumb_up'
]

GESTURES = ['rest', 'fist', 'open', 'point', 'pinch', 'thumb-up']
GESTURE_MAP = {g: i for i, g in enumerate(GESTURES)}
INV_GESTURE_MAP = {i: g for i, g in enumerate(GESTURES)}

SEQ_LEN = 160
WINDOW_SEC = 1.4
TRAIN_STRIDE_SEC = 0.40
BATCH_SIZE = 64
EPOCHS = 32
LR = 1.2e-3
N_FOLDS = 5

DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.startswith('train_') and f.endswith('.xlsx') for f in files):
        DATA_DIR = os.path.dirname(root) if os.path.basename(root) == 'train' else root
        break
if DATA_DIR is None:
    DATA_DIR = '/kaggle/input/competitions/vr-calibration-free-emg-challenge-pieeg'

def natural_key(path):
    nums = re.findall(r'\d+', os.path.basename(path))
    return int(nums[0]) if nums else 0

def read_aura_excel_fast(fpath):
    xl = pd.ExcelFile(fpath, engine='openpyxl')
    sheet = 'multimodal' if 'multimodal' in xl.sheet_names else xl.sheet_names[0]
    df = xl.parse(sheet)
    df.columns = [str(c).replace('#', '').strip() for c in df.columns]
    return fpath, df

# =====================================================================
# 1. SPATIAL-PRESERVING SIGNAL PROCESSING & RIEMANNIAN FEATURES
# =====================================================================
B_HP, A_HP = butter(4, 15.0 / (250.0 / 2.0), btype='highpass')

def extract_v5_window_data(df_win, seq_len=SEQ_LEN):
    """
    CRITICAL FIX OVER V1-V4:
    Never normalizes individual EMG or IMU channels independently by their own window std!
    Instead, normalizes EMG as a joint 8-channel spatial group and Accel/Gyro as 3D vector groups,
    preserving 100% of which specific electrode or IMU axis was active!
    """
    raw = df_win[SIGNAL_COLS].astype(np.float32).values  # (T, 14)
    raw = pd.DataFrame(raw).interpolate(method='linear', limit_direction='both').fillna(0.0).values
    T = raw.shape[0]

    # 1. Mean-center per 1.4s window + remove linear drift
    centered = raw - np.mean(raw, axis=0, keepdims=True)
    if T > 4:
        t_axis = np.linspace(-1.0, 1.0, T, dtype=np.float32)[:, None]
        slope = np.sum(centered * t_axis, axis=0, keepdims=True) / (np.sum(t_axis ** 2) + 1e-8)
        detrended = centered - slope * t_axis
    else:
        detrended = centered

    emg_raw = detrended[:, :8]
    if T > 18:
        emg_hp = filtfilt(B_HP, A_HP, emg_raw.astype(np.float64), axis=0).astype(np.float32)
    else:
        emg_hp = emg_raw

    acc = detrended[:, 8:11]
    gyro = detrended[:, 11:14]

    # First-order temporal derivatives
    emg_diff = np.diff(emg_hp, axis=0, prepend=emg_hp[:1, :])
    acc_diff = np.diff(acc, axis=0, prepend=acc[:1, :])
    gyro_diff = np.diff(gyro, axis=0, prepend=gyro[:1, :])

    # Smooth rectified muscle force envelopes (8 channels)
    win_k = max(5, T // 14)
    emg_env = np.sqrt(pd.DataFrame(emg_hp ** 2).rolling(window=win_k, min_periods=1, center=True).mean().values + 1e-6)

    # --- A. SPATIAL-PRESERVING GROUP NORMALIZATION FOR DEEP LEARNING ---
    # Divide all 8 EMG channels by a SINGLE scalar (the joint 8-channel RMS of the window).
    # Now an active electrode is ~2.5 while a quiet electrode stays ~0.1!
    joint_emg_scale = np.sqrt(np.mean(emg_hp ** 2)) + 1e-4
    joint_diff_scale = np.sqrt(np.mean(emg_diff ** 2)) + 1e-4
    joint_env_scale = np.mean(emg_env) + 1e-4
    joint_acc_scale = np.sqrt(np.mean(acc ** 2)) + 1e-4
    joint_gyro_scale = np.sqrt(np.mean(gyro ** 2)) + 1e-4

    emg_spatial_norm = np.clip(emg_hp / joint_emg_scale, -8.0, 8.0)
    diff_spatial_norm = np.clip(emg_diff / joint_diff_scale, -8.0, 8.0)
    env_spatial_norm = np.clip(emg_env / joint_env_scale, 0.0, 8.0)

    # Also keep log-modulus raw envelope so the network knows absolute 'rest' vs 'active' amplitude
    env_log_amp = np.log1p(emg_env)

    acc_spatial_norm = np.clip(acc / joint_acc_scale, -8.0, 8.0)
    gyro_spatial_norm = np.clip(gyro / joint_gyro_scale, -8.0, 8.0)
    acc_log = np.sign(acc) * np.log1p(np.abs(acc) * 10.0)
    gyro_log = np.sign(gyro) * np.log1p(np.abs(gyro) * 10.0)

    # Stack 44 Spatial-Preserving Channels: 8 + 8 + 8 + 8 + 3 + 3 + 3 + 3 = 44
    combined_seq = np.concatenate([
        emg_spatial_norm, diff_spatial_norm, env_spatial_norm, env_log_amp,
        acc_spatial_norm, gyro_spatial_norm, acc_log, gyro_log
    ], axis=1)

    old_t = np.linspace(0.0, 1.0, T)
    new_t = np.linspace(0.0, 1.0, seq_len)
    seq_resampled = np.zeros((seq_len, combined_seq.shape[1]), dtype=np.float32)
    for c in range(combined_seq.shape[1]):
        seq_resampled[:, c] = np.interp(new_t, old_t, combined_seq[:, c])

    seq_tensor = seq_resampled.T.astype(np.float32)  # (44, 160)

    # --- B. RICH BIOMECHANICAL & RIEMANNIAN TANGENT SPACE FEATURES ---
    feats = []

    # 1. Per-channel EMG Amplitudes & Complexity (48 features)
    ch_rms = np.sqrt(np.mean(emg_hp ** 2, axis=0)) + 1e-5
    ch_mav = np.mean(np.abs(emg_hp), axis=0) + 1e-5
    ch_wl = np.mean(np.abs(emg_diff), axis=0) + 1e-5
    ch_p90 = np.percentile(np.abs(emg_hp), 90, axis=0) + 1e-5
    ch_p50 = np.percentile(np.abs(emg_hp), 50, axis=0) + 1e-5
    ch_zc = np.sum(np.diff(np.sign(emg_hp), axis=0) != 0, axis=0) / float(T)

    feats.extend(np.log1p(ch_rms))
    feats.extend(np.log1p(ch_mav))
    feats.extend(np.log1p(ch_wl))
    feats.extend(np.log1p(ch_p90))
    feats.extend(np.log1p(ch_p50))
    feats.extend(ch_zc)

    # 2. Calibration-Free Spatial Proportions across the 8 Electrodes (32 features)
    rms_prop = ch_rms / np.sum(ch_rms)
    wl_prop = ch_wl / np.sum(ch_wl)
    p90_prop = ch_p90 / np.sum(ch_p90)
    mav_prop = ch_mav / np.sum(ch_mav)
    feats.extend(rms_prop)
    feats.extend(wl_prop)
    feats.extend(p90_prop)
    feats.extend(mav_prop)

    # Global Muscle Energy & Spatial Concentration (8 features)
    tot_rms_log = float(np.log1p(np.sum(ch_rms)))
    tot_wl_log = float(np.log1p(np.sum(ch_wl)))
    entropy = float(-np.sum(rms_prop * np.log(rms_prop + 1e-6)))
    feats.extend([
        tot_rms_log, tot_wl_log, entropy,
        float(np.max(rms_prop)), float(np.min(rms_prop)),
        float(np.std(rms_prop)), float(np.max(rms_prop) / (np.median(rms_prop) + 1e-4)),
        float(np.argmax(rms_prop))
    ])

    # 3. Circular Forearm Ring Antagonist Contrasts (24 features)
    for i in range(8):
        feats.append(float(rms_prop[i] - rms_prop[(i + 4) % 8]))  # Opposing muscle
        feats.append(float(rms_prop[i] - rms_prop[(i + 1) % 8]))  # Neighboring muscle
        feats.append(float(rms_prop[i] - rms_prop[(i + 2) % 8]))  # Orthogonal muscle

    # 4. Riemannian Log-Euclidean Spatial Covariance Matrix (44 features)
    # Normalizing by joint_emg_scale preserves true spatial power on the diagonal AND co-activation off-diagonal!
    cov_spatial = (emg_spatial_norm.T @ emg_spatial_norm) / float(T) + 1e-3 * np.eye(8)
    eigvals, eigvecs = np.linalg.eigh(cov_spatial)
    eigvals = np.clip(eigvals, 1e-4, None)
    log_cov = eigvecs @ np.diag(np.log(eigvals)) @ eigvecs.T
    feats.extend(log_cov[np.triu_indices(8, k=0)])  # 36 upper-triangle + diagonal elements
    feats.extend(np.log(eigvals))                   # 8 sorted spatial synergy eigenvalues

    # 5. Temporal Envelope Quantiles & Sub-Window Profiles (36 features)
    for q_sub in np.array_split(env_spatial_norm, 3, axis=0):
        q_mean = np.mean(q_sub, axis=0)
        feats.extend(q_mean / (np.sum(q_mean) + 1e-5))
    feats.extend(np.percentile(env_spatial_norm, 90, axis=0) - np.percentile(env_spatial_norm, 20, axis=0))
    feats.extend(np.mean(env_log_amp[T // 3 : 2 * T // 3], axis=0) - np.mean(env_log_amp[:T // 3], axis=0))

    # 6. FFT Spectral Power Distribution (24 features)
    fft_p = np.abs(np.fft.rfft(emg_hp, axis=0)) ** 2
    if len(fft_p) > 6:
        b1, b2, b3 = np.array_split(fft_p[1:], 3, axis=0)
        p1, p2, p3 = np.mean(b1, axis=0) + 1e-5, np.mean(b2, axis=0) + 1e-5, np.mean(b3, axis=0) + 1e-5
        ptot = p1 + p2 + p3
        feats.extend(p1 / ptot)
        feats.extend(p2 / ptot)
        feats.extend(p3 / ptot)
    else:
        feats.extend(np.zeros(24))

    # 7. IMU 3D Kinematics, Direction Proportions & Cross-Correlation (55 features)
    imu_full = detrended[:, 8:]
    imu_rms = np.sqrt(np.mean(imu_full ** 2, axis=0)) + 1e-5
    imu_p90 = np.percentile(np.abs(imu_full), 90, axis=0)
    imu_min = np.percentile(imu_full, 10, axis=0)
    imu_max = np.percentile(imu_full, 90, axis=0)

    # Directional proportions (Which axis is rotating/accelerating the most?)
    acc_rms_prop = imu_rms[:3] / (np.sum(imu_rms[:3]) + 1e-5)
    gyro_rms_prop = imu_rms[3:] / (np.sum(imu_rms[3:]) + 1e-5)
    feats.extend(np.log1p(imu_rms))
    feats.extend(np.log1p(imu_p90))
    feats.extend(imu_min)
    feats.extend(imu_max)
    feats.extend(acc_rms_prop)
    feats.extend(gyro_rms_prop)

    # Temporal trajectory (First half vs Second half vs Cumulative displacement)
    imu_traj = np.cumsum(imu_full, axis=0) / float(T)
    feats.extend(imu_traj[-1, :])
    feats.extend(np.mean(imu_full[T // 2:], axis=0) - np.mean(imu_full[:T // 2], axis=0))

    acc_mag = np.sqrt(np.sum(acc ** 2, axis=1))
    gyro_mag = np.sqrt(np.sum(gyro ** 2, axis=1))
    feats.extend([
        float(np.log1p(np.mean(acc_mag))), float(np.log1p(np.std(acc_mag))),
        float(np.log1p(np.mean(gyro_mag))), float(np.log1p(np.std(gyro_mag))),
        float(np.mean(acc_mag) / (np.mean(gyro_mag) + 1e-4))
    ])

    imu_corr = np.nan_to_num(np.corrcoef(imu_full.T), nan=0.0)
    feats.extend(imu_corr[np.triu_indices(6, k=1)])

    tab_vec = np.nan_to_num(np.array(feats, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return seq_tensor, tab_vec

# =====================================================================
# 2. PER-SESSION SUBJECT ALIGNMENT (SOLVES UNSEEN USER SHIFT!)
# =====================================================================
def apply_per_session_alignment(X_seq, X_tab, session_ids):
    """
    Because every file (train_1..28 and test_1..5) is a full recording session of ONE participant,
    we compute each participant's session mean and std and create Session-Aligned Features!
    This guarantees that 'rest' is always negative (-1.5 SD) and 'fist' is always positive (+1.3 SD)
    for EVERY unseen user!
    """
    N, D = X_tab.shape
    X_tab_sess_norm = np.zeros_like(X_tab)
    X_tab_sess_rank = np.zeros((N, 22), dtype=np.float32)
    X_seq_aligned = X_seq.copy()

    # Key amplitude columns to rank-order within each session (8 EMG RMS, 8 EMG WL, tot_rms, tot_wl, 4 IMU)
    rank_cols = list(range(24))[:22]

    for s_id in np.unique(session_ids):
        mask = (session_ids == s_id)
        if np.sum(mask) < 3:
            continue

        # 1. Within-Session Z-Score of all 271 Tabular Features
        sub_tab = X_tab[mask]
        mu = np.mean(sub_tab, axis=0, keepdims=True)
        sd = np.std(sub_tab, axis=0, keepdims=True) + 1e-4
        X_tab_sess_norm[mask] = np.clip((sub_tab - mu) / sd, -6.0, 6.0)

        # 2. Within-Session Empirical Percentile Ranks [0.0 -> 1.0]
        for idx_r, col_r in enumerate(rank_cols):
            vals = sub_tab[:, col_r]
            ranks = np.argsort(np.argsort(vals)).astype(np.float32) / float(max(1, len(vals) - 1))
            X_tab_sess_rank[mask, idx_r] = ranks - 0.5

        # 3. Within-Session Alignment of the Log-Amplitude Sequence Channels (channels 24..44)
        sub_seq = X_seq[mask, 24:, :]
        s_mu = np.mean(sub_seq, axis=(0, 2), keepdims=True)
        s_sd = np.std(sub_seq, axis=(0, 2), keepdims=True) + 1e-4
        X_seq_aligned[mask, 24:, :] = np.clip((sub_seq - s_mu) / s_sd, -6.0, 6.0)

    # Concatenate: [Raw Spatial/Biomechanical Features + Session-Z-Scored Features + Session-Rank Features]
    X_tab_combined = np.concatenate([X_tab, X_tab_sess_norm, X_tab_sess_rank], axis=1)
    return X_seq_aligned, X_tab_combined

# =====================================================================
# 3. PARALLEL DATASET BUILDER
# =====================================================================
def load_v5_datasets(data_dir):
    train_files = sorted(glob.glob(os.path.join(data_dir, "**", "train_*.xlsx"), recursive=True), key=natural_key)
    test_files = sorted(glob.glob(os.path.join(data_dir, "**", "test_*.xlsx"), recursive=True), key=natural_key)
    print(f"Parallel-loading {len(train_files)} train sessions and {len(test_files)} test sessions...")

    with ThreadPoolExecutor(max_workers=4) as ex:
        train_loaded = list(ex.map(read_aura_excel_fast, train_files))
        test_loaded = list(ex.map(read_aura_excel_fast, test_files))

    X_seq, X_tab, y_cls, y_reg, reg_mask, groups, is_anchor = [], [], [], [], [], [], []

    for session_idx, (fpath, df) in enumerate(train_loaded):
        if 'label_source' in df.columns:
            df = df[df['label_source'].astype(str).str.strip().str.lower() == 'guided'].reset_index(drop=True)

        label_col = 'label' if 'label' in df.columns else 'gesture'
        df[label_col] = df[label_col].astype(str).str.strip().str.lower()
        df = df[df[label_col].isin(GESTURE_MAP.keys())].reset_index(drop=True)
        if len(df) == 0:
            continue

        time_jump = df['t'].diff().abs() > 0.25
        label_change = df[label_col] != df[label_col].shift()
        df['seg_id'] = (time_jump | label_change).cumsum()

        for _, seg in df.groupby('seg_id'):
            t_start, t_end = seg['t'].iloc[0], seg['t'].iloc[-1]
            gesture_id = GESTURE_MAP[seg[label_col].iloc[0]]

            curr_t = t_start
            step_idx = 0
            while curr_t + WINDOW_SEC <= t_end + 0.08:
                win = seg[(seg['t'] >= curr_t) & (seg['t'] < curr_t + WINDOW_SEC)]
                if len(win) >= 35:
                    s_t, t_v = extract_v5_window_data(win, seq_len=SEQ_LEN)

                    r_t = np.zeros(len(AUX_REG_COLS), dtype=np.float32)
                    r_m = np.zeros(len(AUX_REG_COLS), dtype=np.float32)
                    for idx, col in enumerate(AUX_REG_COLS):
                        if col in win.columns:
                            vals = pd.to_numeric(win[col], errors='coerce').dropna().values
                            if len(vals) > 0:
                                r_t[idx] = float(np.mean(vals))
                                r_m[idx] = 1.0

                    X_seq.append(s_t)
                    X_tab.append(t_v)
                    y_cls.append(gesture_id)
                    y_reg.append(r_t)
                    reg_mask.append(r_m)
                    groups.append(session_idx)
                    is_anchor.append(1 if (step_idx % 3 == 0) else 0)
                    step_idx += 1
                curr_t += TRAIN_STRIDE_SEC

    X_seq = np.stack(X_seq)
    X_tab = np.stack(X_tab)
    y_cls = np.array(y_cls, dtype=np.int64)
    y_reg = np.stack(y_reg)
    reg_mask = np.stack(reg_mask)
    groups = np.array(groups, dtype=np.int64)
    is_anchor = np.array(is_anchor, dtype=np.int64)

    for j in range(y_reg.shape[1]):
        valid = reg_mask[:, j] > 0.5
        if np.sum(valid) > 10:
            y_reg[valid, j] = np.clip(
                (y_reg[valid, j] - np.mean(y_reg[valid, j])) / (np.std(y_reg[valid, j]) + 1e-5),
                -4.0, 4.0
            )

    # Load Test Sessions (Tracking test_session_ids 0..4 for Per-Session Alignment!)
    X_test_seq, X_test_tab, test_ids, test_sessions = [], [], [], []
    for test_sess_idx, (fpath, df) in enumerate(test_loaded):
        id_col = 'action_id' if 'action_id' in df.columns else df.columns[0]
        for act_id, win in df.groupby(id_col, sort=False):
            s_t, t_v = extract_v5_window_data(win, seq_len=SEQ_LEN)
            X_test_seq.append(s_t)
            X_test_tab.append(t_v)
            test_ids.append(act_id)
            test_sessions.append(test_sess_idx)

    X_test_seq = np.stack(X_test_seq)
    X_test_tab = np.stack(X_test_tab)
    test_sessions = np.array(test_sessions, dtype=np.int64)

    # Apply Per-Session Subject Alignment to both Train Sessions (0..27) and Test Sessions (0..4)
    X_seq, X_tab = apply_per_session_alignment(X_seq, X_tab, groups)
    X_test_seq, X_test_tab = apply_per_session_alignment(X_test_seq, X_test_tab, test_sessions)

    print(f"Train Windows: {len(X_seq)} | Seq: {X_seq.shape} | Session-Aligned Tabular: {X_tab.shape}")
    print(f"Test Actions:  {len(X_test_seq)} across {len(np.unique(test_sessions))} test sessions")
    return X_seq, X_tab, y_cls, y_reg, reg_mask, groups, is_anchor, X_test_seq, X_test_tab, test_ids, test_sessions

# =====================================================================
# 4. TWO DIVERSE SPATIAL-PRESERVING DEEP ARCHITECTURES
# =====================================================================
class AuraV5Dataset(Dataset):
    def __init__(self, X_seq, X_tab, y_cls=None, y_reg=None, reg_mask=None, is_train=True):
        self.X_seq = X_seq
        self.X_tab = X_tab
        self.y_cls = y_cls
        self.y_reg = y_reg
        self.reg_mask = reg_mask
        self.is_train = is_train

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        x = self.X_seq[idx].copy()
        tab = self.X_tab[idx].copy()

        if self.is_train:
            if random.random() < 0.55:
                shift = random.randint(-8, 8)
                x = np.roll(x, shift=shift, axis=-1)
            if random.random() < 0.45:
                scale = np.random.uniform(0.90, 1.10, size=(x.shape[0], 1)).astype(np.float32)
                noise = np.random.normal(0.0, 0.02, size=x.shape).astype(np.float32)
                x = x * scale + noise
            if random.random() < 0.30:
                t0 = random.randint(0, x.shape[1] - 16)
                x[:, t0 : t0 + 16] = 0.0

        if self.y_cls is None:
            return torch.tensor(x, dtype=torch.float32), torch.tensor(tab, dtype=torch.float32)

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(tab, dtype=torch.float32),
            torch.tensor(self.y_cls[idx], dtype=torch.long),
            torch.tensor(self.y_reg[idx], dtype=torch.float32),
            torch.tensor(self.reg_mask[idx], dtype=torch.float32)
        )

class SpatialPreservingConformerNet(nn.Module):
    """
    Model 1: Uses GroupNorm (NEVER InstanceNorm1d!) so relative spatial electrode amplitudes
    are 100% preserved, fused with BiGRU + Self-Attention + Session-Aligned Tabular MLP.
    """
    def __init__(self, in_channels=44, tab_dim=564, d_model=128, num_classes=6, num_reg=10):
        super().__init__()
        self.conv_k3 = nn.Conv1d(in_channels, d_model // 2, kernel_size=3, padding=1)
        self.conv_k7 = nn.Conv1d(in_channels, d_model // 4, kernel_size=7, padding=3)
        self.conv_k15 = nn.Conv1d(in_channels, d_model // 4, kernel_size=15, padding=7)

        self.cnn = nn.Sequential(
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.MaxPool1d(2, 2),  # 160 -> 80
            nn.Conv1d(d_model, d_model, kernel_size=5, padding=2),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.MaxPool1d(2, 2)   # 80 -> 40
        )

        self.bigru = nn.GRU(d_model, d_model // 2, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=d_model * 2,
            dropout=0.2, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=1)

        self.attn_pool = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim),
            nn.Dropout(0.25),
            nn.Linear(tab_dim, 160),
            nn.BatchNorm1d(160),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(160, 96),
            nn.BatchNorm1d(96),
            nn.GELU()
        )

        fused_dim = d_model * 2 + 96
        self.reg_head = nn.Linear(fused_dim, num_reg)
        self.cls_head = nn.Sequential(
            nn.LayerNorm(fused_dim + num_reg),
            nn.Dropout(0.30),
            nn.Linear(fused_dim + num_reg, 128),
            nn.GELU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_seq, x_tab):
        h = torch.cat([self.conv_k3(x_seq), self.conv_k7(x_seq), self.conv_k15(x_seq)], dim=1)
        h = self.cnn(h).permute(0, 2, 1)  # (B, 40, 128)
        seq_g, _ = self.bigru(h)
        seq = self.transformer(seq_g + h)

        w = F.softmax(self.attn_pool(seq), dim=1)
        attn_p = torch.sum(seq * w, dim=1)
        max_p = torch.max(seq, dim=1)[0]
        tab_e = self.tab_mlp(x_tab)

        fused = torch.cat([attn_p, max_p, tab_e], dim=-1)
        reg_p = self.reg_head(fused)
        logits = self.cls_head(torch.cat([fused, reg_p], dim=-1))
        return logits, reg_p

class InceptionWaveBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        mid = out_c // 4
        self.b1 = nn.Conv1d(in_c, mid, kernel_size=1)
        self.b2 = nn.Conv1d(in_c, mid, kernel_size=5, padding=2)
        self.b3 = nn.Conv1d(in_c, mid, kernel_size=11, padding=5)
        self.b4 = nn.Conv1d(in_c, mid, kernel_size=23, padding=11)
        self.bn = nn.BatchNorm1d(out_c)
        self.short = nn.Conv1d(in_c, out_c, kernel_size=1) if in_c != out_c else nn.Identity()

    def forward(self, x):
        out = torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)
        return F.gelu(self.bn(out) + self.short(x))

class MultiScaleInceptionWaveNet(nn.Module):
    """
    Model 2: InceptionTime Multi-Receptive-Field Architecture + Global Average/Std/Max Pooling.
    Excels at time-series gesture classification with zero positional center bias!
    """
    def __init__(self, in_channels=44, tab_dim=564, num_classes=6, num_reg=10):
        super().__init__()
        self.inc1 = InceptionWaveBlock(in_channels, 96)
        self.pool1 = nn.AvgPool1d(2, 2)
        self.inc2 = InceptionWaveBlock(96, 128)
        self.pool2 = nn.AvgPool1d(2, 2)
        self.inc3 = InceptionWaveBlock(128, 128)

        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim),
            nn.Dropout(0.25),
            nn.Linear(tab_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(128, 96),
            nn.GELU()
        )

        # Mean + Std + Max pooling across time = 128 * 3 = 384 + 96 = 480
        fused_dim = 128 * 3 + 96
        self.reg_head = nn.Linear(fused_dim, num_reg)
        self.cls_head = nn.Sequential(
            nn.LayerNorm(fused_dim + num_reg),
            nn.Dropout(0.30),
            nn.Linear(fused_dim + num_reg, 128),
            nn.GELU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_seq, x_tab):
        h = self.pool1(self.inc1(x_seq))
        h = self.pool2(self.inc2(h))
        h = self.inc3(h)  # (B, 128, 40)

        mean_p = h.mean(dim=-1)
        std_p = h.std(dim=-1)
        max_p = h.max(dim=-1)[0]
        tab_e = self.tab_mlp(x_tab)

        fused = torch.cat([mean_p, std_p, max_p, tab_e], dim=-1)
        reg_p = self.reg_head(fused)
        logits = self.cls_head(torch.cat([fused, reg_p], dim=-1))
        return logits, reg_p

class AuraV5Loss(nn.Module):
    def __init__(self, num_classes=6, smoothing=0.05, f1_w=0.35, reg_w=0.20):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.num_classes = num_classes
        self.f1_w = f1_w
        self.reg_w = reg_w

    def forward(self, logits, y_cls, reg_p, y_reg, reg_m):
        ce_loss = self.ce(logits, y_cls)
        probs = F.softmax(logits, dim=-1)
        one_hot = F.one_hot(y_cls, num_classes=self.num_classes).float()
        tp = torch.sum(probs * one_hot, dim=0)
        fp = torch.sum(probs * (1.0 - one_hot), dim=0)
        fn = torch.sum((1.0 - probs) * one_hot, dim=0)
        soft_f1 = (2.0 * tp + 1e-5) / (2.0 * tp + fp + fn + 1e-5)
        loss = (1.0 - self.f1_w) * ce_loss + self.f1_w * (1.0 - torch.mean(soft_f1))

        valid = torch.sum(reg_m)
        if valid > 0:
            reg_l1 = F.smooth_l1_loss(reg_p, y_reg, reduction='none')
            loss = loss + self.reg_w * (torch.sum(reg_l1 * reg_m) / (valid + 1e-5))
        return loss

def predict_nn_tta(model, loader, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for batch in loader:
            bx_s, bx_t = batch[0].to(device), batch[1].to(device)
            l0, _ = model(bx_s, bx_t)
            l1, _ = model(torch.roll(bx_s, shifts=-5, dims=-1), bx_t)
            l2, _ = model(torch.roll(bx_s, shifts=5, dims=-1), bx_t)
            p = 0.50 * F.softmax(l0, dim=-1) + 0.25 * F.softmax(l1, dim=-1) + 0.25 * F.softmax(l2, dim=-1)
            probs_list.append(p.cpu().numpy())
    return np.concatenate(probs_list, axis=0)

# =====================================================================
# 5. TRANSDUCTIVE PER-SESSION SELF-CALIBRATION (SINKHORN + EM PROTOTYPES)
# =====================================================================
def transductive_session_self_calibration(probs, X_tab_sess, session_ids, temp=0.15, n_iter=3, blend_alpha=0.35):
    """
    Calibrates predictions for each unseen user session (val session or test_k.xlsx):
      1. Sinkhorn-Knopp Optimal Transport removes per-user class bias (e.g. over-predicting pinch vs point).
      2. Computes the user's personal 6 gesture centroids in session-whitened feature space
         and refines probabilities using Cosine similarity to that user's own gesture clusters!
    """
    calibrated_probs = probs.copy()
    # Use the session-normalized biomechanical feature block (columns 271..542)
    D_raw = (X_tab_sess.shape[1] - 22) // 2
    Z_sess = X_tab_sess[:, D_raw : 2 * D_raw].copy()
    Z_norm = Z_sess / (np.linalg.norm(Z_sess, axis=1, keepdims=True) + 1e-6)

    for s_id in np.unique(session_ids):
        mask = (session_ids == s_id)
        if np.sum(mask) < 12:
            continue

        P = calibrated_probs[mask].copy() + 1e-6
        Z_s = Z_norm[mask]

        # Step 1: Soft Sinkhorn-Knopp Marginal Balancing within Session s
        for _ in range(4):
            P = P / np.sum(P, axis=1, keepdims=True)
            col_mean = np.mean(P, axis=0, keepdims=True)
            # Softly nudge session class distribution toward balanced prior
            P = P * ((1.0 / 6.0) / (col_mean + 1e-6)) ** 0.5
            P = P / np.sum(P, axis=1, keepdims=True)

        # Step 2: Iterative User-Specific Prototype EM Refinement
        for _ in range(n_iter):
            # Confidence-sharpened weights (gamma = 2.0 focuses on clear examples of each gesture)
            W = (P ** 2.0)
            W = W / (np.sum(W, axis=0, keepdims=True) + 1e-6)
            centroids = W.T @ Z_s  # (6, D)
            centroids = centroids / (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-6)

            # Cosine similarity of every action in Session s to User s's 6 personal centroids
            cos_sim = Z_s @ centroids.T  # (N_s, 6)
            proto_probs = np.exp(cos_sim / temp)
            proto_probs = proto_probs / np.sum(proto_probs, axis=1, keepdims=True)

            P = (1.0 - blend_alpha) * P + blend_alpha * proto_probs
            # Re-balance marginals softly
            col_mean = np.mean(P, axis=0, keepdims=True)
            P = P * ((1.0 / 6.0) / (col_mean + 1e-6)) ** 0.35
            P = P / np.sum(P, axis=1, keepdims=True)

        calibrated_probs[mask] = P

    return calibrated_probs

# =====================================================================
# 6. 5-FOLD EXECUTION: 2 DEEP NETS + 2 GBDTS + TRANSDUCTIVE CALIBRATION
# =====================================================================
(X_seq, X_tab, y_cls, y_reg, reg_mask, groups, is_anchor,
 X_test_seq, X_test_tab, test_ids, test_sessions) = load_v5_datasets(DATA_DIR)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS)

oof_m1 = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_m2 = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_lgb = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_cat = np.zeros((len(X_seq), 6), dtype=np.float32)

test_m1 = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_m2 = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_lgb = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_cat = np.zeros((len(X_test_seq), 6), dtype=np.float32)

criterion = AuraV5Loss()

for fold, (trn_idx, val_idx) in enumerate(sgkf.split(X_seq, y_cls, groups)):
    print(f"\n==================== FOLD {fold + 1} / {N_FOLDS} ====================")
    clean_val_idx = val_idx[is_anchor[val_idx] == 1]
    anchor_submask = (is_anchor[val_idx] == 1)

    scaler = StandardScaler()
    X_tab_trn = np.clip(scaler.fit_transform(X_tab[trn_idx]), -8.0, 8.0)
    X_tab_val = np.clip(scaler.transform(X_tab[val_idx]), -8.0, 8.0)
    X_tab_val_clean = np.clip(scaler.transform(X_tab[clean_val_idx]), -8.0, 8.0)
    X_tab_tst = np.clip(scaler.transform(X_test_tab), -8.0, 8.0)

    # --- 1. Feature Selection & Pruned LightGBM on Session-Aligned Features ---
    scout = LGBMClassifier(
        n_estimators=200, learning_rate=0.04, max_depth=4, num_leaves=15,
        subsample=0.8, colsample_bytree=0.6, random_state=42 + fold, n_jobs=-1, verbose=-1
    )
    scout.fit(X_tab_trn, y_cls[trn_idx])
    keep_k = int(X_tab.shape[1] * 0.75)
    keep_cols = np.argsort(scout.feature_importances_)[::-1][:keep_k]

    lgb_model = LGBMClassifier(
        n_estimators=1400, learning_rate=0.025, max_depth=5, num_leaves=22,
        min_child_samples=30, min_split_gain=0.01, subsample=0.80, subsample_freq=1,
        colsample_bytree=0.60, reg_alpha=2.0, reg_lambda=4.0,
        random_state=100 + fold, n_jobs=-1, verbose=-1
    )
    lgb_model.fit(
        X_tab_trn[:, keep_cols], y_cls[trn_idx],
        eval_set=[(X_tab_val_clean[:, keep_cols], y_cls[clean_val_idx])],
        callbacks=[early_stopping(90, verbose=False)]
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_tab_val[:, keep_cols])
    test_lgb += lgb_model.predict_proba(X_tab_tst[:, keep_cols]) / N_FOLDS
    lgb_f1 = f1_score(y_cls[clean_val_idx], np.argmax(oof_lgb[clean_val_idx], axis=1), average='macro')

    # --- 2. Pruned CatBoost on Session-Aligned Features ---
    if HAS_CATBOOST:
        cat_model = CatBoostClassifier(
            iterations=1400, learning_rate=0.035, depth=5, l2_leaf_reg=6.0,
            random_seed=100 + fold, task_type='CPU', verbose=False
        )
        cat_model.fit(
            X_tab_trn[:, keep_cols], y_cls[trn_idx],
            eval_set=(X_tab_val_clean[:, keep_cols], y_cls[clean_val_idx]),
            early_stopping_rounds=90, verbose=False
        )
        oof_cat[val_idx] = cat_model.predict_proba(X_tab_val[:, keep_cols])
        test_cat += cat_model.predict_proba(X_tab_tst[:, keep_cols]) / N_FOLDS
        cat_f1 = f1_score(y_cls[clean_val_idx], np.argmax(oof_cat[clean_val_idx], axis=1), average='macro')
    else:
        oof_cat[val_idx] = oof_lgb[val_idx]
        test_cat += test_lgb / N_FOLDS
        cat_f1 = lgb_f1

    print(f"  [Session-Aligned Trees] LGBM Val F1: {lgb_f1:.4f} | CatBoost Val F1: {cat_f1:.4f}")

    # DataLoaders for Deep Models
    trn_ds = AuraV5Dataset(X_seq[trn_idx], X_tab_trn, y_cls[trn_idx], y_reg[trn_idx], reg_mask[trn_idx], is_train=True)
    val_ds = AuraV5Dataset(X_seq[val_idx], X_tab_val, is_train=False)
    tst_ds = AuraV5Dataset(X_test_seq, X_tab_tst, is_train=False)

    trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    tst_loader = DataLoader(tst_ds, batch_size=BATCH_SIZE, shuffle=False)

    # --- 3. Train Model 1: SpatialPreservingConformerNet ---
    m1 = SpatialPreservingConformerNet(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
    opt1 = torch.optim.AdamW(m1.parameters(), lr=LR, weight_decay=1.5e-2)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=EPOCHS, eta_min=1e-5)
    best_m1_f1, best_m1_p, ckpt1 = 0.0, None, f"v5_m1_fold{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        m1.train()
        for bx_s, bx_t, by_c, by_r, bm in trn_loader:
            bx_s, bx_t = bx_s.to(DEVICE), bx_t.to(DEVICE)
            by_c, by_r, bm = by_c.to(DEVICE), by_r.to(DEVICE), bm.to(DEVICE)
            opt1.zero_grad()
            logits, reg_p = m1(bx_s, bx_t)
            loss = criterion(logits, by_c, reg_p, by_r, bm)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m1.parameters(), 1.0)
            opt1.step()
        sch1.step()

        vp = predict_nn_tta(m1, val_loader, DEVICE)
        cf1 = f1_score(y_cls[val_idx][anchor_submask], np.argmax(vp[anchor_submask], axis=1), average='macro')
        if cf1 > best_m1_f1:
            best_m1_f1, best_m1_p = cf1, vp
            torch.save(m1.state_dict(), ckpt1)

    oof_m1[val_idx] = best_m1_p
    m1.load_state_dict(torch.load(ckpt1, map_location=DEVICE))
    test_m1 += predict_nn_tta(m1, tst_loader, DEVICE) / N_FOLDS

    # --- 4. Train Model 2: MultiScaleInceptionWaveNet ---
    m2 = MultiScaleInceptionWaveNet(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
    opt2 = torch.optim.AdamW(m2.parameters(), lr=LR, weight_decay=1.5e-2)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS, eta_min=1e-5)
    best_m2_f1, best_m2_p, ckpt2 = 0.0, None, f"v5_m2_fold{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        m2.train()
        for bx_s, bx_t, by_c, by_r, bm in trn_loader:
            bx_s, bx_t = bx_s.to(DEVICE), bx_t.to(DEVICE)
            by_c, by_r, bm = by_c.to(DEVICE), by_r.to(DEVICE), bm.to(DEVICE)
            opt2.zero_grad()
            logits, reg_p = m2(bx_s, bx_t)
            loss = criterion(logits, by_c, reg_p, by_r, bm)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m2.parameters(), 1.0)
            opt2.step()
        sch2.step()

        vp = predict_nn_tta(m2, val_loader, DEVICE)
        cf1 = f1_score(y_cls[val_idx][anchor_submask], np.argmax(vp[anchor_submask], axis=1), average='macro')
        if cf1 > best_m2_f1:
            best_m2_f1, best_m2_p = cf1, vp
            torch.save(m2.state_dict(), ckpt2)

    oof_m2[val_idx] = best_m2_p
    m2.load_state_dict(torch.load(ckpt2, map_location=DEVICE))
    test_m2 += predict_nn_tta(m2, tst_loader, DEVICE) / N_FOLDS

    print(f"  [Deep Models] Conformer Val F1: {best_m1_f1:.4f} | InceptionWaveNet Val F1: {best_m2_f1:.4f}")

# =====================================================================
# 7. ENSEMBLE BLENDING + TRANSDUCTIVE SELF-CALIBRATION VERIFICATION
# =====================================================================
clean_mask = (is_anchor == 1)
y_clean = y_cls[clean_mask]

f1_m1 = f1_score(y_clean, np.argmax(oof_m1[clean_mask], axis=1), average='macro')
f1_m2 = f1_score(y_clean, np.argmax(oof_m2[clean_mask], axis=1), average='macro')
f1_lgb = f1_score(y_clean, np.argmax(oof_lgb[clean_mask], axis=1), average='macro')
f1_cat = f1_score(y_clean, np.argmax(oof_cat[clean_mask], axis=1), average='macro')

# Optimize 4-Model Blend Weights on Clean Anchor OOF
best_raw_f1 = 0.0
best_w = (0.30, 0.25, 0.25, 0.20)

for w1 in np.linspace(0.10, 0.50, 9):
    for w2 in np.linspace(0.10, 0.50, 9):
        for w3 in np.linspace(0.10, 0.50, 9):
            w4 = max(0.0, 1.0 - w1 - w2 - w3)
            if w1 + w2 + w3 > 1.0:
                continue
            blend = w1 * oof_m1[clean_mask] + w2 * oof_m2[clean_mask] + w3 * oof_lgb[clean_mask] + w4 * oof_cat[clean_mask]
            s = f1_score(y_clean, np.argmax(blend, axis=1), average='macro')
            if s > best_raw_f1:
                best_raw_f1 = s
                best_w = (w1, w2, w3, w4)

oof_blend_raw = best_w[0] * oof_m1 + best_w[1] * oof_m2 + best_w[2] * oof_lgb + best_w[3] * oof_cat
test_blend_raw = best_w[0] * test_m1 + best_w[1] * test_m2 + best_w[2] * test_lgb + best_w[3] * test_cat

# Apply Transductive Per-Session Self-Calibration & Verify Improvement on OOF before using on Test!
best_cal_f1 = best_raw_f1
best_cal_params = (0.0, 0.20)
oof_blend_cal = oof_blend_raw.copy()

for alpha in [0.15, 0.25, 0.35, 0.45]:
    for temp in [0.12, 0.18, 0.25]:
        cand_oof = transductive_session_self_calibration(
            oof_blend_raw, X_tab, groups, temp=temp, n_iter=3, blend_alpha=alpha
        )
        s = f1_score(y_clean, np.argmax(cand_oof[clean_mask], axis=1), average='macro')
        if s > best_cal_f1:
            best_cal_f1 = s
            best_cal_params = (alpha, temp)
            oof_blend_cal = cand_oof

if best_cal_params[0] > 0.0:
    test_blend_final = transductive_session_self_calibration(
        test_blend_raw, X_test_tab, test_sessions,
        temp=best_cal_params[1], n_iter=3, blend_alpha=best_cal_params[0]
    )
else:
    test_blend_final = test_blend_raw

# Final Conservative Class Multiplier Tuning
class_mult = np.ones(6, dtype=np.float32)
tuned_f1 = best_cal_f1
for _ in range(3):
    for c in range(6):
        for m in np.linspace(0.85, 1.20, 36):
            cand = class_mult.copy()
            cand[c] = m
            s = f1_score(y_clean, np.argmax(oof_blend_cal[clean_mask] * cand[None, :], axis=1), average='macro')
            if s > tuned_f1:
                tuned_f1 = s
                class_mult = cand

print("\n========================================================")
print(f"Clean OOF Macro-F1 (SpatialConformer):       {f1_m1:.4f}")
print(f"Clean OOF Macro-F1 (InceptionWaveNet):       {f1_m2:.4f}")
print(f"Clean OOF Macro-F1 (Session-Aligned LGBM):   {f1_lgb:.4f}")
print(f"Clean OOF Macro-F1 (Session-Aligned CatB):   {f1_cat:.4f}")
print(f"Clean OOF Macro-F1 (4-Model Ensemble Raw):   {best_raw_f1:.4f} | Weights: {np.round(best_w, 2)}")
print(f"Clean OOF Macro-F1 (+Session Self-Calib):    {best_cal_f1:.4f} | Alpha/Temp: {best_cal_params}")
print(f"Clean OOF Macro-F1 (+Tuned Multipliers):     {tuned_f1:.4f}")
print("========================================================")
print(classification_report(y_clean, np.argmax(oof_blend_cal[clean_mask] * class_mult[None, :], axis=1), target_names=GESTURES))

# Generate & Save Final submission.csv
final_preds = np.argmax(test_blend_final * class_mult[None, :], axis=1)
final_labels = [INV_GESTURE_MAP[i] for i in final_preds]

sample_sub_files = glob.glob(os.path.join(DATA_DIR, "**", "sample_submission*.csv"), recursive=True)
if sample_sub_files:
    sub = pd.read_csv(sample_sub_files[0])
    id_col, target_col = sub.columns[0], sub.columns[1]
    if len(sub) == len(final_labels) and len(set(test_ids)) == len(test_ids):
        pred_map = dict(zip(test_ids, final_labels))
        sub[target_col] = sub[id_col].map(pred_map).fillna(pd.Series(final_labels))
    else:
        sub[target_col] = final_labels
else:
    sub = pd.DataFrame({'action_id': test_ids, 'gesture': final_labels})

sub.to_csv("submission.csv", index=False)
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("\nSaved V5 submission.csv! Test Set Prediction Distribution:")
print(sub.iloc[:, 1].value_counts())